# LYS v2 architecture comparator — Kaggle P100

This notebook compares a transparent **An et al. (2023)-inspired 3-D U-Net** with **nnU-Net v2** on the exact preserved five grouped LYS development folds. It then tests external-mouse initialization only for the direct-target winner.

It never regenerates target folds and never materializes or evaluates the 57-case locked test. If the old locked test has already been opened, it cannot validate architecture choices made here; a new untouched final test is required. Predictions remain draft masks.

Important limitation: the Nature paper's public repository is inference-only. The first model is therefore a clearly labelled paper-inspired full-field adaptation, not an exact reproduction. See `docs/lys_v2_architecture_comparator_protocol.md` for the scientific contract and source links.


## 1 — Configuration and exact software setup

Enable a P100 GPU and Internet. Keep `NNUNET_TRAINER="nnUNetTrainer_250epochs"` for the practical compute-budget screen. Set it to `"nnUNetTrainer"` **before the first run** for canonical 1,000-epoch nnU-Net. Never mix trainer budgets across folds. `FOLDS_TO_RUN` can be shortened per Kaggle session; completed folds are skipped and interrupted folds resume.


In [ ]:
import hashlib
import importlib.metadata
import json
import os
import shutil
import subprocess
import sys
import tarfile
import zipfile
from pathlib import Path

import pandas as pd
import torch
from IPython.display import FileLink, Image, display

RUN_SEED = 20260715
BRANCH = "dl-ratlesnetv2-finetune"
REPOSITORY = "https://github.com/paulaize/LYS_PROJ1.git"
NNUNET_COMMIT = "468cf803df9b267150ae2b6c0c59b8ac84f16227"  # v2.8.1
NNUNET_TRAINER = "nnUNetTrainer_250epochs"  # or nnUNetTrainer before starting
NNUNET_PLANNER = "nnUNetPlannerResEncM"      # official 9-11 GB preset
NNUNET_PLANS = "nnUNetResEncUNetMPlans"
NNUNET_CONFIGURATION = "3d_fullres"
PAPER_EPOCHS = 200
FOLDS_TO_RUN = [0, 1, 2, 3, 4]
RUN_EXTERNAL_STAGE = True
PRUNE_COMPLETED_NNUNET_NONSELECTED_CHECKPOINTS = True
BUILD_FULL_RESUME_ARCHIVE = True
SPLIT_ASSIGNMENTS_OVERRIDE = None  # optional absolute Kaggle path
RESUME_ARCHIVE_OVERRIDE = None     # optional absolute Kaggle path
RATLES_CE_METRICS_OVERRIDE = None  # optional selected_threshold_case_metrics.csv

assert set(FOLDS_TO_RUN) <= set(range(5)) and len(FOLDS_TO_RUN) == len(set(FOLDS_TO_RUN))
assert NNUNET_TRAINER in {"nnUNetTrainer_250epochs", "nnUNetTrainer"}

WORK = Path("/kaggle/working")
PROJECT = WORK / "LYS_PROJ1"
if not (PROJECT / ".git").is_dir():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPOSITORY, str(PROJECT)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT), "pull", "--ff-only"], check=True)
PROJECT_COMMIT = subprocess.check_output(
    ["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True
).strip()
required = [
    PROJECT / "ratlesnetv2_finetune/scripts/prepare_architecture_comparator.py",
    PROJECT / "ratlesnetv2_finetune/scripts/train_an2023_unet_adapted.py",
    PROJECT / "ratlesnetv2_finetune/scripts/compare_oof_candidates.py",
    PROJECT / "ratlesnetv2_finetune/scripts/create_oof_qc_contact_sheet.py",
]
assert all(path.is_file() for path in required), (
    "The branch lacks the architecture-comparator code. "
    "Push/pull the new commit first."
)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     f"git+https://github.com/MIC-DKFZ/nnUNet.git@{NNUNET_COMMIT}",
     "nibabel", "pandas", "scipy", "scikit-image", "matplotlib"],
    check=True,
)
sys.path.insert(0, str(PROJECT))

assert importlib.metadata.version("nnunetv2") == "2.8.1"
assert torch.cuda.is_available(), "Enable a Kaggle GPU before continuing"
GPU_NAME = torch.cuda.get_device_name(0)
print("GPU:", GPU_NAME)
if "P100" not in GPU_NAME.upper():
    print("WARNING: this protocol was budgeted for a P100; record the actual GPU in provenance.")
subprocess.run(["nvidia-smi"], check=True)
for command in ("nnUNetv2_plan_and_preprocess", "nnUNetv2_train",
                "nnUNetv2_move_plans_between_datasets"):
    assert shutil.which(command), command
    subprocess.run([command, "--help"], check=True, stdout=subprocess.DEVNULL)

COMPARATOR_ROOT = WORK / "LYS_v2_architecture_comparator"
NNUNET_RAW = WORK / "nnUNet_raw"
NNUNET_PREPROCESSED = WORK / "nnUNet_preprocessed"
NNUNET_RESULTS = WORK / "nnUNet_results"
for key, value in {"nnUNet_raw": NNUNET_RAW, "nnUNet_preprocessed": NNUNET_PREPROCESSED,
                   "nnUNet_results": NNUNET_RESULTS}.items():
    os.environ[key] = str(value)

TARGET_ID = 701
TARGET_DATASET = "Dataset701_LYSDevelopmentV1"
EXTERNAL_ID = 702
EXTERNAL_DATASET = "Dataset702_ExternalMouseV0"
PAPER_DIRECT = "an2023_adapted_direct"
budget_name = "250" if NNUNET_TRAINER.endswith("250epochs") else "1000"
NNUNET_DIRECT = f"nnunetv2_resencm_{budget_name}_direct"
print("Project commit:", PROJECT_COMMIT)
print("Direct candidates:", PAPER_DIRECT, NNUNET_DIRECT)


## 2 — Restore a previous comparator session, if attached

Attach at most one prior `LYS_v2_architecture_comparator_resume.tar.gz`. The archive contains run/checkpoint state, not raw datasets; raw conversion and preprocessing are rebuilt and verified. Skip this automatically on the first session.


In [ ]:
def safe_extract(archive_path: Path, destination: Path) -> None:
    root = destination.resolve()
    with tarfile.open(archive_path, "r:*") as archive:
        for member in archive.getmembers():
            target = (destination / member.name).resolve()
            assert target == root or root in target.parents, member.name
        archive.extractall(destination)

if RESUME_ARCHIVE_OVERRIDE:
    resume_archives = [Path(RESUME_ARCHIVE_OVERRIDE)]
else:
    resume_archives = sorted(
        Path("/kaggle/input").rglob("LYS_v2_architecture_comparator_resume.tar.gz")
    )
assert len(resume_archives) <= 1, resume_archives
if resume_archives:
    assert not COMPARATOR_ROOT.exists() and not NNUNET_RESULTS.exists(), (
        "Refusing to merge an attached resume archive into existing run state"
    )
    safe_extract(resume_archives[0], WORK)
    print("Restored:", resume_archives[0])
else:
    print("No resume archive attached; starting or continuing the current Kaggle working session.")

COMPARATOR_ROOT.mkdir(parents=True, exist_ok=True)
PROVENANCE = COMPARATOR_ROOT / "provenance"
PROVENANCE.mkdir(exist_ok=True)
identity = {
    "protocol": "lys_v2_architecture_comparator_v1",
    "project_commit": PROJECT_COMMIT,
    "run_seed": RUN_SEED,
    "gpu_name": GPU_NAME,
    "paper_epochs": PAPER_EPOCHS,
    "nnunet_version": importlib.metadata.version("nnunetv2"),
    "nnunet_source_commit": NNUNET_COMMIT,
    "nnunet_trainer": NNUNET_TRAINER,
    "nnunet_planner": NNUNET_PLANNER,
    "nnunet_plans": NNUNET_PLANS,
    "nnunet_configuration": NNUNET_CONFIGURATION,
}
identity_path = PROVENANCE / "protocol_identity.json"
if identity_path.exists():
    assert json.loads(identity_path.read_text()) == identity, (
        "Restored outputs were made with a different protocol/software identity"
    )
else:
    identity_path.write_text(json.dumps(identity, indent=2, sort_keys=True) + "\n")
identity


## 3 — Locate and normalize both prepared datasets

This is the same payload-aware repair used by the RatLesNetV2 notebook. It handles Kaggle's plain `.nii` and truncated gzip suffixes without renaming source files blindly.


In [ ]:
def locate_or_extract_prepared_dataset(dataset_name: str) -> Path:
    input_root = Path("/kaggle/input")
    matches = sorted(input_root.rglob(f"{dataset_name}/manifest.csv"))
    if len(matches) == 1:
        return matches[0].parent
    assert not matches, f"Multiple exposed {dataset_name} roots: {matches}"
    archives = sorted(input_root.rglob(f"{dataset_name}.tar.gz"))
    assert len(archives) == 1, f"Expected one {dataset_name} archive; found {archives}"
    extract_root = WORK / "uploaded_archives" / dataset_name
    extract_root.mkdir(parents=True, exist_ok=True)
    safe_extract(archives[0], extract_root)
    matches = sorted(extract_root.rglob(f"{dataset_name}/manifest.csv"))
    assert len(matches) == 1, matches
    return matches[0].parent

NORMALIZED_BASE = WORK / "normalized_inputs"
def normalize(source: Path, name: str, count: int) -> Path:
    subprocess.run(
        [sys.executable, "-m", "ratlesnetv2_finetune.scripts.normalize_prepared_dataset",
         "--input", str(source), "--output-base", str(NORMALIZED_BASE),
         "--dataset-name", name, "--expected-cases", str(count)],
        cwd=PROJECT, check=True,
    )
    result = NORMALIZED_BASE / name
    assert len(list(result.rglob("scan.nii.gz"))) == count
    return result

LYS_NAME = "LYS_T2w_manual_v1"
EXTERNAL_NAME = "External_Mouse_T2w_manual_LSP_SI_v0"
LYS_ROOT = normalize(locate_or_extract_prepared_dataset(LYS_NAME), LYS_NAME, 258)
EXTERNAL_ROOT = normalize(locate_or_extract_prepared_dataset(EXTERNAL_NAME), EXTERNAL_NAME, 426)
print("LYS_ROOT:", LYS_ROOT)
print("EXTERNAL_ROOT:", EXTERNAL_ROOT)


## 4 — Restore and verify the exact RatLesNetV2 v1 split assignments

This cell intentionally has no split-generation fallback. Attach the previous RatLesNetV2 artifact bundle or point `SPLIT_ASSIGNMENTS_OVERRIDE` to its CSV. Duplicate copies are allowed only when their SHA-256 hashes are identical.


In [ ]:
def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def valid_target_split(path: Path):
    try:
        rows = pd.read_csv(path, keep_default_na=False)
    except Exception:
        return None
    needed = {"case_id", "subject_id", "outer_split", "cv_fold"}
    if len(rows) != 258 or not needed <= set(rows.columns) or not rows.case_id.is_unique:
        return None
    if (rows.outer_split == "development").sum() != 201 or (rows.outer_split == "test").sum() != 57:
        return None
    return rows

candidate_paths = []
if SPLIT_ASSIGNMENTS_OVERRIDE:
    candidate_paths.append(Path(SPLIT_ASSIGNMENTS_OVERRIDE))
else:
    candidate_paths.extend(Path("/kaggle/input").rglob("split_assignments.csv"))
    current = WORK / "LYS_v1_grouped/split_assignments.csv"
    if current.is_file():
        candidate_paths.append(current)

archive_candidates = sorted(Path("/kaggle/input").rglob("*RatLesNetV2*artifacts*.tar.gz"))
extracted_candidates = PROVENANCE / "split_candidates"
extracted_candidates.mkdir(exist_ok=True)
for archive_index, archive_path in enumerate(archive_candidates):
    with tarfile.open(archive_path, "r:*") as archive:
        members = [
            member for member in archive.getmembers()
            if member.isfile() and member.name.endswith("split_assignments.csv")
        ]
        for member_index, member in enumerate(members):
            handle = archive.extractfile(member)
            assert handle is not None
            output = extracted_candidates / f"archive_{archive_index}_{member_index}.csv"
            output.write_bytes(handle.read())
            candidate_paths.append(output)

accepted = [(path, valid_target_split(path)) for path in candidate_paths if path.is_file()]
accepted = [(path, rows) for path, rows in accepted if rows is not None]
assert accepted, ("No preserved 258-row LYS v1 split_assignments.csv found. "
                  "Attach the prior RatLesNetV2 artifact; do not regenerate the split.")
hash_groups = {}
for path, rows in accepted:
    hash_groups.setdefault(sha256(path), []).append((path, rows))
assert len(hash_groups) == 1, f"Conflicting preserved LYS split files: {list(hash_groups)}"
split_hash, copies = next(iter(hash_groups.items()))
SPLIT_ASSIGNMENTS = PROVENANCE / "split_assignments.csv"
if SPLIT_ASSIGNMENTS.exists():
    assert sha256(SPLIT_ASSIGNMENTS) == split_hash
else:
    shutil.copy2(copies[0][0], SPLIT_ASSIGNMENTS)
assignments = pd.read_csv(SPLIT_ASSIGNMENTS, keep_default_na=False)
development = assignments[assignments.outer_split == "development"].copy()
locked = assignments[assignments.outer_split == "test"].copy()
assert assignments.groupby("subject_id").outer_split.nunique().max() == 1
assert development.groupby("subject_id").cv_fold.nunique().max() == 1
assert set(development.cv_fold.astype(str)) == {"0", "1", "2", "3", "4"}
assert set(locked.cv_fold) == {""}
lys_manifest = pd.read_csv(LYS_ROOT / "manifest.csv")
assert set(lys_manifest.case_id) == set(assignments.case_id)
print("Preserved split SHA-256:", split_hash)
display(pd.crosstab([assignments.outer_split, assignments.cv_fold], assignments.cohort))


## 5 — Build development-only inputs for both architectures

The paper-inspired folds use symlinks and consume negligible extra storage. The nnU-Net raw dataset contains only the 201 development cases. Both conversion reports explicitly record that 57 locked cases were omitted.


In [ ]:
PAPER_FOLDS = COMPARATOR_ROOT / "paper_folds"
TARGET_RAW = NNUNET_RAW / TARGET_DATASET
subprocess.run(
    [sys.executable, "-m", "ratlesnetv2_finetune.scripts.prepare_architecture_comparator",
     "prepare-paper-folds", "--input", str(LYS_ROOT),
     "--split-assignments", str(SPLIT_ASSIGNMENTS), "--output", str(PAPER_FOLDS)],
    cwd=PROJECT, check=True,
)
subprocess.run(
    [sys.executable, "-m", "ratlesnetv2_finetune.scripts.prepare_architecture_comparator",
     "prepare-target", "--input", str(LYS_ROOT),
     "--split-assignments", str(SPLIT_ASSIGNMENTS), "--output", str(TARGET_RAW),
     "--dataset-id", str(TARGET_ID), "--dataset-name", "LYSDevelopmentV1"],
    cwd=PROJECT, check=True,
)
paper_report = json.loads((PAPER_FOLDS / "conversion_report.json").read_text())
target_report = json.loads((TARGET_RAW / "conversion_report.json").read_text())
assert paper_report["n_locked_test_omitted"] == target_report["n_locked_test_omitted"] == 57
assert target_report["n_training"] == 201 and target_report["locked_test_materialized"] is False
display(paper_report)
display(target_report)


## 6 — Plan and preprocess nnU-Net ResEnc M

Planning is data-driven, but splitting is not: after preprocessing the exact preserved `splits_final.json` is installed in nnU-Net's required location and verified byte-for-byte as JSON.


In [ ]:
TARGET_PREPROCESSED = NNUNET_PREPROCESSED / TARGET_DATASET
preprocess_marker = PROVENANCE / "target_nnunet_preprocessing_complete.json"
plan_path = TARGET_PREPROCESSED / f"{NNUNET_PLANS}.json"
previous_plan_hash = (json.loads(preprocess_marker.read_text())["plans_sha256"]
                      if preprocess_marker.is_file() else None)
if not preprocess_marker.is_file() or not plan_path.is_file():
    subprocess.run(
        ["nnUNetv2_plan_and_preprocess", "-d", str(TARGET_ID),
         "-pl", NNUNET_PLANNER, "-c", NNUNET_CONFIGURATION,
         "-npfp", "4", "-np", "4", "--verify_dataset_integrity"],
        check=True,
    )
    assert plan_path.is_file()
    if previous_plan_hash is not None:
        assert sha256(plan_path) == previous_plan_hash, (
            "Rebuilt target plans differ from resumed run"
        )
    preprocess_marker.write_text(json.dumps({"plans_sha256": sha256(plan_path)}, indent=2) + "\n")
else:
    assert plan_path.is_file()
    assert json.loads(preprocess_marker.read_text())["plans_sha256"] == sha256(plan_path)
shutil.copy2(TARGET_RAW / "splits_final.json", TARGET_PREPROCESSED / "splits_final.json")
assert json.loads((TARGET_PREPROCESSED / "splits_final.json").read_text()) == json.loads(
    (TARGET_RAW / "splits_final.json").read_text()
)
plans = json.loads(plan_path.read_text())
configuration = plans["configurations"][NNUNET_CONFIGURATION]
display({key: configuration.get(key) for key in
         ("data_identifier", "median_image_size_in_voxels", "spacing", "patch_size",
          "batch_size", "architecture")})


## 7 — Restart-safe training and OOF helpers

A paper-inspired run resumes from `last_checkpoint.pth`. An nnU-Net run resumes with `--c`, validates the best checkpoint with `--npz --val_best`, asserts the expected fold IDs, converts probabilities to native orientation, and may then delete only the unselected final/latest nnU-Net checkpoints to save space. `checkpoint_best.pth` is always retained.


In [ ]:
RUNS = COMPARATOR_ROOT / "runs"
THRESHOLDS = COMPARATOR_ROOT / "thresholds"
COMPARISONS = COMPARATOR_ROOT / "comparisons"
for path in (RUNS, THRESHOLDS, COMPARISONS):
    path.mkdir(exist_ok=True)

def paper_train(train: Path, validation: Path, output: Path, seed: int, pretrained=None):
    status_path = output / "run_status.json"
    manifest = output / "prediction_exports/validation/final/prediction_export_manifest.csv"
    if status_path.is_file() and manifest.is_file():
        status = json.loads(status_path.read_text())["status"]
        if status in {"completed", "early_stopped"}:
            return output / "an2023_adapted_unet.model"
    command = [sys.executable, "-m", "ratlesnetv2_finetune.scripts.train_an2023_unet_adapted",
               "--input", str(train), "--validation", str(validation),
               "--output", str(output), "--epochs", str(PAPER_EPOCHS),
               "--early-stop-patience", "30", "--early-stop-min-delta", "0.001",
               "--learning-rate", "1e-4", "--gradient-accumulation", "8",
               "--noise-std", "0.45", "--seed", str(seed), "--device", "cuda"]
    if pretrained is not None:
        command.extend(["--pretrained-model", str(pretrained)])
    if (output / "last_checkpoint.pth").is_file():
        command.append("--resume")
    subprocess.run(command, cwd=PROJECT, check=True)
    assert manifest.is_file()
    return output / "an2023_adapted_unet.model"

target_mapping = pd.read_csv(TARGET_RAW / "case_mapping.csv", keep_default_na=False)
def nnunet_model_folder(dataset_name: str, trainer: str, plans_name: str, fold) -> Path:
    return (NNUNET_RESULTS / dataset_name /
            f"{trainer}__{plans_name}__{NNUNET_CONFIGURATION}" / f"fold_{fold}")

def nnunet_train(dataset_id: int, dataset_name: str, fold, plans_name: str, marker: Path,
                  expected_ids, candidate=None, export_fold=None, pretrained=None) -> Path:
    folder = nnunet_model_folder(dataset_name, NNUNET_TRAINER, plans_name, fold)
    best = folder / "checkpoint_best.pth"
    validation = folder / "validation"
    if marker.is_file():
        recorded = json.loads(marker.read_text())
        assert best.is_file() and recorded["checkpoint_best_sha256"] == sha256(best)
        return best
    base = ["nnUNetv2_train", str(dataset_id), NNUNET_CONFIGURATION, str(fold),
            "-tr", NNUNET_TRAINER, "-p", plans_name]
    final = folder / "checkpoint_final.pth"
    if final.is_file():
        subprocess.run(base + ["--val", "--npz", "--val_best"], check=True)
    else:
        command = base.copy()
        if (folder / "checkpoint_latest.pth").is_file():
            command.append("--c")
        elif pretrained is not None:
            command.extend(["-pretrained_weights", str(pretrained)])
        command.extend(["--npz", "--val_best"])
        subprocess.run(command, check=True)
    assert best.is_file() and validation.is_dir()
    observed_ids = {path.stem for path in validation.glob("*.npz")}
    assert observed_ids == set(expected_ids), (
        f"Validation IDs differ: missing={set(expected_ids)-observed_ids}, "
        f"unexpected={observed_ids-set(expected_ids)}"
    )
    if candidate is not None:
        export_root = (RUNS / candidate / f"fold_{export_fold}" /
                       "prediction_exports/validation/final")
        if not (export_root / "prediction_export_manifest.csv").is_file():
            subprocess.run(
                [sys.executable, "-m",
                 "ratlesnetv2_finetune.scripts.prepare_architecture_comparator",
                 "export-validation", "--validation", str(validation),
                 "--case-mapping", str(TARGET_RAW / "case_mapping.csv"),
                 "--fold", str(export_fold), "--output", str(export_root),
                 "--candidate", candidate], cwd=PROJECT, check=True,
            )
    marker.parent.mkdir(parents=True, exist_ok=True)
    marker.write_text(json.dumps({"checkpoint_best": str(best),
                                  "checkpoint_best_sha256": sha256(best)},
                                 indent=2, sort_keys=True) + "\n")
    if PRUNE_COMPLETED_NNUNET_NONSELECTED_CHECKPOINTS:
        for disposable in (folder / "checkpoint_latest.pth", final):
            if disposable.is_file():
                disposable.unlink()
    return best

def require_oof(candidate: str):
    manifests = []
    for fold in range(5):
        path = (RUNS / candidate / f"fold_{fold}" /
                "prediction_exports/validation/final/prediction_export_manifest.csv")
        assert path.is_file(), f"Missing completed {candidate} fold {fold}: {path}"
        manifests.append(path)
    rows = pd.concat([pd.read_csv(path) for path in manifests], ignore_index=True)
    assert len(rows) == 201 and rows.case_id.is_unique
    assert set(rows.case_id) == set(development.case_id)
    assert set(rows.split) == {"validation"}
    return manifests

def calibrate(candidate: str):
    require_oof(candidate)
    output = THRESHOLDS / candidate
    subprocess.run(
        [sys.executable, "-m", "ratlesnetv2_finetune.scripts.calibrate_probability_threshold",
         "--prediction-root", str(RUNS / candidate), "--metadata", str(SPLIT_ASSIGNMENTS),
         "--output", str(output), "--thresholds", "0.20:0.80:0.05",
         "--surface-tolerance-mm", "0.2", "--bootstrap-samples", "2000",
         "--seed", str(RUN_SEED), "--overwrite"], cwd=PROJECT, check=True,
    )
    return json.loads((output / "selected_threshold.json").read_text())

def compare(reference: str, candidate: str, name: str):
    output = COMPARISONS / name
    if output.exists():
        shutil.rmtree(output)
    subprocess.run(
        [sys.executable, "-m", "ratlesnetv2_finetune.scripts.compare_oof_candidates",
         "--reference-name", reference,
         "--reference", str(THRESHOLDS / reference / "selected_threshold_case_metrics.csv"),
         "--candidate-name", candidate,
         "--candidate", str(THRESHOLDS / candidate / "selected_threshold_case_metrics.csv"),
         "--output", str(output), "--bootstrap-samples", "2000",
         "--seed", str(RUN_SEED)], cwd=PROJECT, check=True,
    )
    return json.loads((output / "paired_summary.json").read_text())


## 8 — Train the five direct An et al.-inspired folds

If a Kaggle session ends, attach the resume archive made in Cell 15, rerun Cells 1–7, and rerun this cell. Completed folds skip; an interrupted fold resumes.


In [ ]:
for fold in FOLDS_TO_RUN:
    fold_root = PAPER_FOLDS / f"fold_{fold}"
    paper_train(fold_root / "train", fold_root / "validation",
                RUNS / PAPER_DIRECT / f"fold_{fold}", RUN_SEED + fold)
print("Requested paper-inspired folds complete.")


In [ ]:
paper_selection = calibrate(PAPER_DIRECT)
display(paper_selection)


## 9 — Train the five direct nnU-Net folds

A full five-fold `nnUNetTrainer_250epochs` screen is still substantial on a P100. Use `FOLDS_TO_RUN` and the resume artifact across sessions. Canonical `nnUNetTrainer` is roughly four times the nominal iteration budget.


In [ ]:
for fold in FOLDS_TO_RUN:
    expected_ids = target_mapping.loc[
        target_mapping.cv_fold.astype(str) == str(fold), "nnunet_case_id"
    ].tolist()
    nnunet_train(
        TARGET_ID, TARGET_DATASET, fold, NNUNET_PLANS,
        RUNS / NNUNET_DIRECT / f"fold_{fold}/nnunet_complete.json",
        expected_ids, candidate=NNUNET_DIRECT, export_fold=fold,
    )
print("Requested nnU-Net folds complete.")


In [ ]:
nnunet_selection = calibrate(NNUNET_DIRECT)
display(nnunet_selection)


## 10 — Paired direct-architecture selection

The predeclared primary rule is the larger pooled mean per-case OOF Dice. The paired 95% bootstrap interval, five fold means, cohort means, failures, HD95, surface Dice, and volume errors remain in the artifact. A CI crossing zero means the architecture advantage is uncertain even though the primary rule still produces a reproducible winner.


In [ ]:
direct_comparison = compare(PAPER_DIRECT, NNUNET_DIRECT, "paper_vs_nnunet_direct")
direct_scores = {PAPER_DIRECT: paper_selection["mean_dice"],
                 NNUNET_DIRECT: nnunet_selection["mean_dice"]}
SELECTED_DIRECT = max(direct_scores, key=direct_scores.get)
direct_decision = {
    "selection_data": "same_201_LYS_OOF_development_cases",
    "primary_rule": "maximum pooled mean per-case OOF Dice",
    "selected_direct_candidate": SELECTED_DIRECT,
    "scores": direct_scores,
    "paired_comparison": direct_comparison,
    "locked_test_used": False,
}
(COMPARISONS / "direct_architecture_selection.json").write_text(
    json.dumps(direct_decision, indent=2, sort_keys=True) + "\n"
)
display(direct_decision)
display(pd.read_csv(COMPARISONS / "paper_vs_nnunet_direct/paired_folds.csv"))
display(pd.read_csv(COMPARISONS / "paper_vs_nnunet_direct/paired_cohorts.csv"))


## 11 — Optional external-source adaptation for the direct winner

The same deterministic 341/85 external animal-grouped split is used for either architecture. The source checkpoint is selected on external validation, then the five unchanged LYS development folds determine whether initialization helps. External validation is not target-domain evidence.


In [ ]:
EXTERNAL_SPLIT = WORK / "External_pretrain_split"
if RUN_EXTERNAL_STAGE:
    subprocess.run(
        [sys.executable, "-m", "ratlesnetv2_finetune.scripts.split_prepared_dataset",
         "--input", str(EXTERNAL_ROOT), "--output", str(EXTERNAL_SPLIT),
         "--group-by", "animal_id", "--validation-fraction", "0.20",
         "--test-fraction", "0", "--seed", str(RUN_SEED),
         "--copy-mode", "symlink", "--overwrite"], cwd=PROJECT, check=True,
    )
    external_manifest = pd.read_csv(EXTERNAL_SPLIT / "manifest.csv")
    assert dict(external_manifest.split.value_counts()) == {"train": 341, "validation": 85}
    assert not (set(external_manifest.loc[external_manifest.split == "train", "animal_id"]) &
                set(external_manifest.loc[external_manifest.split == "validation", "animal_id"]))
    display(external_manifest.split.value_counts())
else:
    print("External stage disabled. Direct OOF comparison is complete.")


In [ ]:
if RUN_EXTERNAL_STAGE and SELECTED_DIRECT == PAPER_DIRECT:
    EXTERNAL_CANDIDATE = "an2023_adapted_external_pretrained"
    source_model = paper_train(
        EXTERNAL_SPLIT / "train", EXTERNAL_SPLIT / "validation",
        RUNS / "an2023_adapted_external_source", RUN_SEED,
    )
    for fold in FOLDS_TO_RUN:
        fold_root = PAPER_FOLDS / f"fold_{fold}"
        paper_train(
            fold_root / "train", fold_root / "validation",
            RUNS / EXTERNAL_CANDIDATE / f"fold_{fold}", RUN_SEED + fold,
            pretrained=source_model,
        )
    print("Requested paper-inspired external-initialized folds complete.")


In [ ]:
if RUN_EXTERNAL_STAGE and SELECTED_DIRECT == NNUNET_DIRECT:
    EXTERNAL_CANDIDATE = f"nnunetv2_resencm_{budget_name}_external_pretrained"
    EXTERNAL_RAW = NNUNET_RAW / EXTERNAL_DATASET
    subprocess.run(
        [sys.executable, "-m", "ratlesnetv2_finetune.scripts.prepare_architecture_comparator",
         "prepare-external", "--input", str(EXTERNAL_SPLIT), "--output", str(EXTERNAL_RAW),
         "--dataset-id", str(EXTERNAL_ID), "--dataset-name", "ExternalMouseV0"],
        cwd=PROJECT, check=True,
    )
    source_plans = f"{NNUNET_PLANS}ExternalSourceV1"
    source_preprocessed = NNUNET_PREPROCESSED / EXTERNAL_DATASET
    source_preprocess_marker = PROVENANCE / "external_nnunet_preprocessing_complete.json"
    source_plan_path = source_preprocessed / f"{source_plans}.json"
    previous_source_plan_hash = (json.loads(source_preprocess_marker.read_text())["plans_sha256"]
                                 if source_preprocess_marker.is_file() else None)
    if not source_preprocess_marker.is_file() or not source_plan_path.is_file():
        subprocess.run(["nnUNetv2_extract_fingerprint", "-d", str(EXTERNAL_ID),
                        "-np", "4", "--verify_dataset_integrity"], check=True)
        subprocess.run(["nnUNetv2_move_plans_between_datasets",
                        "-s", str(TARGET_ID), "-t", str(EXTERNAL_ID),
                        "-sp", NNUNET_PLANS, "-tp", source_plans], check=True)
        subprocess.run(["nnUNetv2_preprocess", "-d", str(EXTERNAL_ID),
                        "-plans_name", source_plans, "-c", NNUNET_CONFIGURATION,
                        "-np", "4"], check=True)
        if previous_source_plan_hash is not None:
            assert sha256(source_plan_path) == previous_source_plan_hash, (
                "Rebuilt external plans differ from resumed run"
            )
        source_preprocess_marker.write_text(
            json.dumps({"plans_sha256": sha256(source_plan_path)}, indent=2) + "\n"
        )
    assert source_plan_path.is_file()
    shutil.copy2(EXTERNAL_RAW / "splits_final.json", source_preprocessed / "splits_final.json")
    external_mapping = pd.read_csv(EXTERNAL_RAW / "case_mapping.csv", keep_default_na=False)
    expected_source_validation = external_mapping.loc[
        external_mapping.source_split == "validation", "nnunet_case_id"
    ].tolist()
    source_best = nnunet_train(
        EXTERNAL_ID, EXTERNAL_DATASET, 0, source_plans,
        RUNS / "nnunet_external_source/nnunet_complete.json",
        expected_source_validation,
    )
    target_external_plans = f"{NNUNET_PLANS}ExternalInitV1"
    target_external_plan_path = TARGET_PREPROCESSED / f"{target_external_plans}.json"
    if not target_external_plan_path.is_file():
        transferred = json.loads(plan_path.read_text())
        transferred["plans_name"] = target_external_plans
        target_external_plan_path.write_text(
            json.dumps(transferred, indent=2, sort_keys=True) + "\n"
        )
    for fold in FOLDS_TO_RUN:
        expected_ids = target_mapping.loc[
            target_mapping.cv_fold.astype(str) == str(fold), "nnunet_case_id"
        ].tolist()
        nnunet_train(
            TARGET_ID, TARGET_DATASET, fold, target_external_plans,
            RUNS / EXTERNAL_CANDIDATE / f"fold_{fold}/nnunet_complete.json",
            expected_ids, candidate=EXTERNAL_CANDIDATE, export_fold=fold,
            pretrained=source_best,
        )
    print("Requested nnU-Net external-initialized folds complete.")


## 12 — Paired initialization selection

This cell requires all five external-initialized target folds. It uses the same primary mean-Dice rule and retains all paired diagnostics.


In [ ]:
if RUN_EXTERNAL_STAGE:
    external_selection = calibrate(EXTERNAL_CANDIDATE)
    initialization_comparison = compare(
        SELECTED_DIRECT, EXTERNAL_CANDIDATE, "direct_vs_external_initialization"
    )
    initialization_scores = {
        SELECTED_DIRECT: json.loads(
            (THRESHOLDS / SELECTED_DIRECT / "selected_threshold.json").read_text()
        )["mean_dice"],
        EXTERNAL_CANDIDATE: external_selection["mean_dice"],
    }
    SELECTED_NEW_CANDIDATE = max(initialization_scores, key=initialization_scores.get)
    initialization_decision = {
        "selection_data": "same_201_LYS_OOF_development_cases",
        "primary_rule": "maximum pooled mean per-case OOF Dice",
        "selected_candidate": SELECTED_NEW_CANDIDATE,
        "scores": initialization_scores,
        "paired_comparison": initialization_comparison,
        "external_validation_is_target_evidence": False,
        "locked_test_used": False,
    }
else:
    SELECTED_NEW_CANDIDATE = SELECTED_DIRECT
    initialization_decision = {"selected_candidate": SELECTED_DIRECT,
                               "external_stage_run": False, "locked_test_used": False}
(COMPARISONS / "initialization_selection.json").write_text(
    json.dumps(initialization_decision, indent=2, sort_keys=True) + "\n"
)
display(initialization_decision)
if RUN_EXTERNAL_STAGE:
    display(pd.read_csv(COMPARISONS / "direct_vs_external_initialization/paired_folds.csv"))
    display(pd.read_csv(COMPARISONS / "direct_vs_external_initialization/paired_cohorts.csv"))


## 13 — Optional paired comparison with the prior RatLesNetV2 CE+Dice winner

Attach or point to the prior `direct_ce_dice/selected_threshold_case_metrics.csv`. Without that exact case-level report, this notebook can select between the two new architectures but cannot scientifically declare that either beats RatLesNetV2.


In [ ]:
if RATLES_CE_METRICS_OVERRIDE:
    ratles_candidates = [Path(RATLES_CE_METRICS_OVERRIDE)]
else:
    ratles_candidates = [
        path
        for path in Path("/kaggle/input").rglob("selected_threshold_case_metrics.csv")
        if path.parent.name == "direct_ce_dice"
    ]
    extracted_ratles = PROVENANCE / "ratles_direct_ce_dice_case_metrics.csv"
    for archive_path in archive_candidates:
        with tarfile.open(archive_path, "r:*") as archive:
            members = [m for m in archive.getmembers() if m.isfile() and
                       m.name.endswith("direct_ce_dice/selected_threshold_case_metrics.csv")]
            for member in members:
                handle = archive.extractfile(member)
                assert handle is not None
                extracted_ratles.write_bytes(handle.read())
                ratles_candidates.append(extracted_ratles)
ratles_candidates = [path for path in ratles_candidates if path.is_file()]
ratles_by_hash = {}
for path in ratles_candidates:
    ratles_by_hash.setdefault(sha256(path), path)
if len(ratles_by_hash) == 1:
    ratles_case_metrics = next(iter(ratles_by_hash.values()))
    ratles_threshold_root = THRESHOLDS / "ratles_direct_ce_dice"
    ratles_threshold_root.mkdir(exist_ok=True)
    shutil.copy2(ratles_case_metrics, ratles_threshold_root / "selected_threshold_case_metrics.csv")
    ratles_vs_new = compare("ratles_direct_ce_dice", SELECTED_NEW_CANDIDATE,
                            "ratles_ce_dice_vs_selected_new")
    display(ratles_vs_new)
elif ratles_candidates:
    raise RuntimeError(f"Conflicting RatLes CE+Dice case-metric files found: {ratles_by_hash}")
else:
    ratles_vs_new = None
    print(
        "Prior RatLes case-level OOF report not attached; "
        "overall architecture decision remains pending."
    )


## 13A — Quick visual QC of prior direct CE+Dice OOF masks

This optional cell displays and saves a PNG comparing T2 scan, manual mask, CE+Dice prediction, and errors at the selected OOF threshold. It shows the four lowest-Dice cases plus four cases spread from Q1 through the best result. It reads development predictions only.


In [ ]:
DIRECT_CE_QC_PNG = WORK / "direct_ce_dice_oof_qc.png"
qc_command = [
    sys.executable, "-m",
    "ratlesnetv2_finetune.scripts.create_oof_qc_contact_sheet",
    "--output", str(DIRECT_CE_QC_PNG), "--cases", "8", "--overwrite",
]
if archive_candidates:
    assert len(archive_candidates) == 1, archive_candidates
    qc_command.extend(
        ["--artifact-bundle", str(archive_candidates[0]),
         "--staging-root", str(WORK / "prior_ratles_direct_ce_qc")]
    )
else:
    current_runs = WORK / "lys_v1_runs/direct_ce_dice"
    current_thresholds = WORK / "lys_v1_thresholds/direct_ce_dice"
    assert current_runs.is_dir(), (
        "Attach the prior RatLes artifact or run this in its original Kaggle session"
    )
    qc_command.extend(
        ["--prediction-root", str(current_runs),
         "--threshold-json", str(current_thresholds / "selected_threshold.json"),
         "--case-metrics",
         str(current_thresholds / "selected_threshold_case_metrics.csv")]
    )
subprocess.run(qc_command, cwd=PROJECT, check=True)
display(Image(filename=str(DIRECT_CE_QC_PNG)))
display(pd.read_csv(DIRECT_CE_QC_PNG.with_suffix(".csv")))
print("Saved Kaggle output PNG:", DIRECT_CE_QC_PNG)


## 14 — Freeze the development-only candidate specification

This is not a locked-test result. It freezes the evidence needed to decide the next phase and records checkpoint hashes, threshold provenance, `postprocessing=none`, and the test boundary. A 250-epoch nnU-Net winner is labelled as requiring canonical-budget confirmation before a definitive nnU-Net claim.


In [ ]:
selected_threshold = json.loads(
    (THRESHOLDS / SELECTED_NEW_CANDIDATE / "selected_threshold.json").read_text()
)
checkpoint_paths = []
if SELECTED_NEW_CANDIDATE.startswith("an2023"):
    checkpoint_paths = [RUNS / SELECTED_NEW_CANDIDATE / f"fold_{fold}/an2023_adapted_unet.model"
                        for fold in range(5)]
else:
    selected_plans = (
        NNUNET_PLANS
        if SELECTED_NEW_CANDIDATE == NNUNET_DIRECT
        else f"{NNUNET_PLANS}ExternalInitV1"
    )
    checkpoint_paths = [nnunet_model_folder(TARGET_DATASET, NNUNET_TRAINER, selected_plans, fold) /
                        "checkpoint_best.pth" for fold in range(5)]
assert all(path.is_file() for path in checkpoint_paths), checkpoint_paths
development_spec = {
    "protocol": "lys_v2_architecture_comparator_v1",
    "selected_new_candidate": SELECTED_NEW_CANDIDATE,
    "selected_probability_threshold": selected_threshold["selected_threshold"],
    "threshold_selection": selected_threshold,
    "ensemble": "mean_probability_across_five_grouped_fold_models",
    "postprocessing": "none",
    "checkpoint_selection": "fold_validation_Dice",
    "checkpoint_sha256": {str(path): sha256(path) for path in checkpoint_paths},
    "split_assignments_sha256": sha256(SPLIT_ASSIGNMENTS),
    "project_commit": PROJECT_COMMIT,
    "nnunet_version": importlib.metadata.version("nnunetv2"),
    "nnunet_source_commit": NNUNET_COMMIT,
    "nnunet_trainer": NNUNET_TRAINER if SELECTED_NEW_CANDIDATE.startswith("nnunet") else None,
    "canonical_nnunet_confirmation_required": (
        SELECTED_NEW_CANDIDATE.startswith("nnunet") and NNUNET_TRAINER == "nnUNetTrainer_250epochs"
    ),
    "locked_test_materialized": False,
    "locked_test_used": False,
    "old_locked_test_reusable_for_new_architecture_selection": False,
    "prior_ratles_paired_comparison_available": ratles_vs_new is not None,
    "model_outputs_are_draft_masks": True,
}
spec_path = COMPARATOR_ROOT / "development_frozen_specification.json"
spec_path.write_text(json.dumps(development_spec, indent=2, sort_keys=True) + "\n")
display(development_spec)


## 15 — Build the compact review bundle and full resume archive

Download the compact ZIP to give another model all decision-relevant tables, logs, plans, protocol text, and hashes. The full tarball retains checkpoints and OOF maps so a later Kaggle session can resume; it can be much larger. Run this cell at the end of every Kaggle session, even if only some folds are complete.


In [ ]:
review_readme = COMPARATOR_ROOT / "README_FOR_MODEL_REVIEW.md"
review_readme.write_text(
    "# LYS v2 architecture-comparator review bundle\n\n"
    "Read `development_frozen_specification.json`, `comparisons/`, and `thresholds/` first. "
    "The locked test was not materialized or evaluated. Check whether the selected "
    "candidate clearly "
    "improves Dice without unacceptable failures, HD95, surface Dice, volume error, or cohort/fold "
    "regression. If nnU-Net used 250 epochs and won, decide whether the gain justifies a five-fold "
    "canonical 1,000-epoch confirmation. If prior RatLes case metrics are absent, "
    "request them before declaring an overall architecture winner. Do not recommend "
    "reusing an already opened locked test.\n"
)

REVIEW_ZIP = WORK / "LYS_v2_architecture_comparator_review.zip"
allowed_suffixes = {".json", ".csv", ".txt", ".md", ".png", ".pdf",
                    ".ipynb", ".yml", ".yaml"}
with zipfile.ZipFile(REVIEW_ZIP, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    roots = [COMPARATOR_ROOT, TARGET_RAW, TARGET_PREPROCESSED, NNUNET_RESULTS]
    for root in roots:
        if not root.exists():
            continue
        for path in sorted(root.rglob("*")):
            if path.is_file() and not path.is_symlink() and path.suffix.lower() in allowed_suffixes:
                if path.stat().st_size <= 25 * 1024 * 1024:
                    archive.write(path, path.relative_to(WORK))
    for qc_path in (WORK / "direct_ce_dice_oof_qc.png",
                    WORK / "direct_ce_dice_oof_qc.csv"):
        if qc_path.is_file():
            archive.write(qc_path, qc_path.relative_to(WORK))
    for path in [PROJECT / "docs/lys_v2_architecture_comparator_protocol.md",
                 PROJECT / "notebooks/lys_v2_architecture_comparator_kaggle.ipynb",
                 *required]:
        archive.write(path, Path("repository_snapshot") / path.relative_to(PROJECT))
print("Compact review bundle:", REVIEW_ZIP, f"{REVIEW_ZIP.stat().st_size / 1e6:.1f} MB")
display(FileLink(str(REVIEW_ZIP)))

if BUILD_FULL_RESUME_ARCHIVE:
    RESUME_ARCHIVE = WORK / "LYS_v2_architecture_comparator_resume.tar.gz"
    with tarfile.open(RESUME_ARCHIVE, "w:gz", compresslevel=1) as archive:
        for root in (COMPARATOR_ROOT, NNUNET_RESULTS):
            if root.exists():
                archive.add(root, arcname=root.relative_to(WORK), recursive=True)
    print("Full resume archive:", RESUME_ARCHIVE, f"{RESUME_ARCHIVE.stat().st_size / 1e9:.2f} GB")
    display(FileLink(str(RESUME_ARCHIVE)))


## End state

Do not add locked-test inference to this notebook. First review the compact artifact, compare the selected new candidate against the prior RatLes CE+Dice OOF report, and—if nnU-Net 250 wins—decide whether to repeat the five nnU-Net folds at the canonical 1,000-epoch budget. Only then define a new final-training/test protocol.
